# ④ 생성 — 질문에 답하기
담당: 생성 · 코드: `rag/generation/generation.py`

03 검증의 `artifacts/eval_set.jsonl` 질문을 재작성해 검색하고, 찾은 청크를 근거로 `gpt-5-mini` 가 원본 질문에 답합니다.
평가셋 순서대로 직전 질문·답변 최대 10개를 참고합니다. 사실의 근거는 현재 검색한 문서로 제한합니다.
답변 · 검색 근거 · 토큰 · 지연 시간과 `rewrite`(검색용 질문·모델·토큰·지연 시간)를 `artifacts/answers.jsonl` 로 저장합니다 → 05 평가가 읽습니다.
`ranked_candidates`는 패킹 전 후보의 chunk_id/doc_id·fused_rank·dense_score/rank·bm25_score/rank·exact_code_match를 기록합니다. `context_chunk_ids`는 실제 생성에 보낸 청크입니다. `retrieval_settings`는 dense_weight·candidate_k·token_budget·활성 chunk_file·k_values(5,10,20)·stored_rewrites_reused(False)를 기록합니다.
`.env` 에 Langfuse 키가 있으면 질문마다 Langfuse Cloud 에 추적 `answer-question`(재작성 `rewrite-question` + 검색 `retrieve-context` + 생성 `generate-answer` — 모델 · 토큰 · 비용)이 남고,
그 추적 id 가 `answers.jsonl` 의 `trace_id` 에 저장되어 05 평가가 점수를 붙입니다 (키가 없으면 기록하지 않고 똑같이 동작).

In [ ]:
import pandas as pd

from rag.config import ARTIFACTS, load_config, read_jsonl, write_jsonl
from rag.generation.generation import answer_questions
from rag.retrieval.retrieval import HybridIndex
from rag.usage import usage_summary

cfg = load_config()
retrieval_settings, = read_jsonl(ARTIFACTS / "retrieval_settings.jsonl")
cfg.update(retrieval_settings)


## 설정
답변 생성의 초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
rewrite 모델·추론 강도와 검색 비율은 02에서 저장한 `artifacts/retrieval_settings.jsonl`을 사용합니다. 먼저 02를 실행해야 하며, 검색 설정을 바꾸려면 02 설정 셀부터 다시 실행하세요.
답변 모델을 바꾸면 `LLM_PRICES`도 해당 모델의 입력 · 캐시 입력 · 출력 단가(USD / 100만 토큰)로 맞추세요. 생성 시 계산한 비용을 저장하며 05는 그 값을 사용합니다.


In [ ]:
LLM_MODEL = cfg["llm"]["model"]                     # 예: "gpt-5-mini"
LLM_REASONING_EFFORT = cfg["llm"]["reasoning_effort"]  # 예: "low"
LLM_MAX_TOKENS = cfg["llm"]["max_tokens"]             # 출력·추론 토큰 한도
LLM_PRICES = cfg["llm"]["usd_per_1m_tokens"]          # 입력 · 캐시 입력 · 출력 단가

cfg["llm"].update(model=LLM_MODEL, reasoning_effort=LLM_REASONING_EFFORT, max_tokens=LLM_MAX_TOKENS,
                  usd_per_1m_tokens=LLM_PRICES)
cfg["llm"]


In [ ]:
eval_set = read_jsonl(ARTIFACTS / "eval_set.jsonl")
index = HybridIndex(ARTIFACTS, cfg)
answers = answer_questions(eval_set, index, cfg["llm"], cfg["rewrite"])
write_jsonl(answers, ARTIFACTS / "answers.jsonl")
pd.DataFrame(answers)[["id", "question", "answer", "context_chunk_ids", "ranked_candidates", "retrieval_settings", "latency_s"]]

## API 사용량
rewrite와 답변 생성 호출이 각각 `artifacts/api_usage.csv`에 자동 추가됩니다. 아래 두 표는 누적 사용량입니다.

In [ ]:
usage_summary("rewrite")

In [ ]:
usage_summary("answer")